# 🌌 Rendering Volume Snapshots 

This notebook walks through the process of rendering Volume Snapshots as examplifed by converting Athena++ MHD simulation snapshots (`.athdf` files) into **OpenVDB volumetric files** for 3D rendering in Blender.

---

## 📦 Pipeline Overview

```
Athena++ Snapshots (.athdf) --> yt (Data Loading) --> AstroVis (Data Processing) --> OpenVDB Volumes (.vdb file) --> Blender

```
The Data Processing Part by AstroVis has to be ran outside Blender with python 3.8.

## 1. Imports Library


In [ ]:
from AstroVis.vdb import *        # VDB export helpers: load_remap_amr_volume, volume_to_multiple_vdbs
import yt                         # Astrophysical data analysis framework
import os                         # Standard file system utilities

## 2. Snapshot Selection

Define where to read snapshots from and where to write VDB frames to.Scan the input directory for `.athdf` files, sort them so frames are in chronological order.


In [ ]:
# --- Paths ---
input_dir  = "M1AD1_snapshots"   # Folder containing Athena++ .athdf snapshot files
output_dir = "M1AD1_vdb"         # Folder where per-frame .vdb files will be written

os.makedirs(output_dir, exist_ok=True)

snapshots = sorted([
    f for f in os.listdir(input_dir)
    if f.endswith(".athdf")
])

print(f"Snapshots Found       : {len(snapshots)}")


## 3. Data Processing

For each snapshot we:

1. **Load** the dataset with `yt.load()`
2. **Remap** the AMR (Adaptive Mesh Refinement) grid  with `load_remap_amr_volume()` to prevent overlapping
3. **Export** the density field as a VDB file with `volume_to_multiple_vdbs()`

### Parameter Notes

| Parameter | Value | Meaning |
|---|---|---|
| `field` | `'density'` | The physical field to volumetrize (mass density ρ) |
| `scale` | `20` | Spatial scale factor applied when writing the VDB grid |
| `file_name_prefix` | `frame_NNN` | Output filename base; the function appends `.vdb` |

In [ ]:
for frame_num, snapshot in enumerate(snapshots):
    
    filepath = os.path.join(input_dir, snapshot)

    # --- Step 1: Load the Athena++ snapshot via yt ---
    ds = yt.load(filepath)

    # --- Step 2: Remap AMR hierarchy → uniform volume array ---
    fh = load_remap_amr_volume(ds)

    # --- Step 3: Write density field to VDB ---
    volume_to_multiple_vdbs(
        fh,
        field='density',
        scale=20,
        file_name_prefix=f"{output_dir}/frame_{frame_num:03d}/frame_{frame_num:03d}" ## Create subfolder for each frame
    )

    print(f" Snapshot {snapshot} is processed to frame_{frame_num:03d}.vdb")

print(f"\n Export complete. {frame_num + 1} VDB frames are written to '{output_dir}/'")

# 4. Setup Animation in Blender

> ⚠️ Run the cells below inside Blender's **Python Console** or **Text Editor** — `bpy` is only available there.

With the VDB files exported, we now move into Blender to assemble them into a rendered animation. The following code runs inside Blender's built-in Python Console or Text Editor as bpy is only available there.

In blender built-in Python Console, we
1. Create a material by create_volume_shaders() 
2. Load VDB Frames as Animation by setup_volume_anaimation

### Parameter Notes

`setup_volume_animation()` scans the VDB folder, creates one Blender Volume object per frame, keyframes their visibility on/off, and assigns the material — producing a complete frame sequence ready to render.

| Parameter | Value | Meaning |
|---|---|---|
| `vdb_folder` | `M1AD1_multi_vdb/` | Folder written by `volume_to_multiple_vdbs()` in Section 3 |
| `multi_vdb_per_frame` | `True` | Multiple `.vdb` files per snapshot (one per AMR level); the helper groups them back into one logical frame |
| `material` | `mat` | `star_volshader` applied to every volume object |

**Meaning of `multi_vdb_per_frame=True`:**
```
frame_000_level0.vdb ┐
frame_000_level1.vdb ├─► Multiple Volume object, visible only at frame 0
frame_000_level2.vdb ┘
frame_001_level0.vdb ┐
frame_001_level1.vdb ├─► Multiple Volume object, visible only at frame 1
frame_001_level2.vdb ┘
```




In [ ]:
import bpy
from AstroVis.api import *  ## Note that it is import from .api rather then .vdb


vdb_folder = r"Your/Path/To/M1AD1_vdb"  # Path to the folder containing the per-frame VDB subfolders

## Step 1: Create a volume shader for the star formation visualization
create_volume_shaders("star")
mat = bpy.data.materials.get("star_volshader") # The name of volume matieral is name + "_volshader"

## Step 2: Set up the volume animation in Blender using the exported VDB files
setup_volume_animation(vdb_folder, multi_vdb_per_frame = True, material = mat)